# Asociación y Localización Sísmica utilizando GaMMA

Este notebook muestra cómo realizar el proceso de asociación de fases sísmicas y localización de eventos utilizando el algoritmo GaMMA, a partir de picks generados previamente con EQTransformer.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from datetime import datetime

from gamma.utils import association

from obspy import UTCDateTime

## Definir los paths de las carpetas importantes

- **`Picks_File`**  
  Archivo de picks completo que contiene todas las estaciones a agrupar y procesar.  
  En este caso, se utilizan picks generados con EQTransformer.

- **`Station_File`**  
  Archivo de estaciones utilizado durante la etapa de detección.  
  Formatos soportados:
  - `CSV` (recomendado)
  - `JSON` (requiere un procesamiento ligeramente diferente)

- **`Output_dir`**  
  Directorio donde se almacenarán los resultados generados, archivos intermedios y salidas finales del procesamiento.

In [ ]:
PICKS_FILE = "/home/ovsicori/datos_sebas/Pacifico_Central_133/RESULTS/2026-133/DETECTION/detection.csv"

STATIONS_FILE = "/home/ovsicori/datos_sebas/Pacifico_Central_133/RESULTS/2026-133/stations/stations.csv"

OUTPUT_DIR = "/home/ovsicori/datos_sebas/Pacifico_Central_133/RESULTS/2026-133/DETECTION/RESULTS_Gamma"

## Estaciones

El proceso de asociación requiere las coordenadas de las estaciones para estimar los tiempos de viaje entre las fuentes sísmicas y cada estación.

El archivo de estaciones debe contener, como mínimo, la siguiente información:

- Código de estación
- Latitud
- Longitud
- Elevación

In [ ]:
stations = pd.read_csv(STATIONS_FILE)

stations["network"] = "OV"

stations["id"] = (
    stations["network"] + "." + stations["code"]
)

stations = stations.rename(columns={
    "latitude": "lat",
    "longitude": "lon",
    "elevation": "elevation(m)"
})

stations.head()

## Conversión de Coordenadas

Importante: el procesamiento utiliza coordenadas en proyección UTM, por lo que es necesario convertir las coordenadas geográficas (`latitude` y `longitude`) antes de ejecutar la asociación.

Además:

- Las coordenadas deben expresarse en kilómetros (`km`)
- La elevación también debe convertirse a kilómetros (`km`)

In [ ]:
from pyproj import Proj

# UTM projection Costa Rica
proj = Proj(proj="utm", zone=16, datum="WGS84")

x, y = proj(
    stations["lon"].values,
    stations["lat"].values
)

stations["x(km)"] = x / 1000
stations["y(km)"] = y / 1000
stations["z(km)"] = -stations["elevation(m)"] / 1000

# para centrar coodenadas en el centro de la red
x0 = stations["x(km)"].mean()
y0 = stations["y(km)"].mean()

stations["x(km)"] -= x0
stations["y(km)"] -= y0


stations.head()

## Picks de EQTransformer

EQTransformer genera picks de manera independiente para cada estación sísmica.

La información típica incluida en los picks es:

- Nombre de la estación
- Tiempo del pick
- Tipo de fase (`P` o `S`)
- Probabilidad de detección

In [ ]:
picks = pd.read_csv(PICKS_FILE)

picks.head()

## Preparación de Picks para GaMMA

Antes de ejecutar la asociación, los picks deben adaptarse al formato esperado por GaMMA.

Durante esta etapa se realiza:

- Conversión de tiempos a formato `datetime`
- Creación de un identificador único de estación (`NETWORK.STATION`)
- Renombrado de columnas para compatibilidad con GaMMA

Columnas utilizadas finalmente:

- `timestamp` → tiempo del pick
- `id` → identificador de estación
- `type` → tipo de fase (`P` o `S`)
- `prob` → probabilidad asociada al pick

In [ ]:
picks["timestamp"] = pd.to_datetime(
    picks["arrival_time"],
    format="mixed"
)

picks["id"] = picks["network"] + "." + picks["station"]

picks = picks.rename(columns={
    "phase": "type",
    "probability": "prob"
})

picks.head()

## Configuración de GaMMA

La siguiente configuración controla el comportamiento del proceso de asociación.


In [ ]:
config = {

    # Rango horizontal en X (km)
    #debe cubrir toda la red sísmica y zona de estudio
    "x(km)": [-250, 250],

    # Rango horizontal en Y (km)
    "y(km)": [-250, 250],

    # Rango de profundidades permitido (km)
    "z(km)": (0, 150),

    "dims": ["x(km)", "y(km)", "z(km)"],

    "vel": {

        # Velocidad de onda P
        "p": 6.3,

        # Velocidad de onda S
        "s": 3.64
    },

    # Método de asociación
    "method": "BGMM",

    # False = solo tiempos de arribo
    "use_amplitude": False,

    # Distancia máxima entre picks para formar clusters iniciales
    "dbscan_eps": 10,

    # Número mínimo de picks para formar un cluster
    "dbscan_min_samples": 3,

    # Número mínimo total de picks para aceptar un evento
    "min_picks_per_eq": 6,

    # Número mínimo de fases P requeridas
    "min_p_picks_per_eq": 3,

    # Número mínimo de fases S requeridas
    "min_s_picks_per_eq": 1,

    # Cantidad de soluciones iniciales exploradas
    "oversample_factor": 10,

    # Control de incertidumbre máxima permitida
    #valores bajos generan catálogos más estrictos
    "max_sigma11": 2.0,

    "max_sigma22": 1.0,
}

## Phase Association


In [ ]:
events, assignments = association(
    picks,
    stations,
    config
)

In [ ]:
events = pd.DataFrame(events)
assignments = pd.DataFrame(assignments)

In [ ]:
events.head()

In [ ]:
proj = Proj(proj="utm", zone=16, datum="WGS84")

x_abs = (events["x(km)"] + x0) * 1000
y_abs = (events["y(km)"] + y0) * 1000

lon, lat = proj(
    x_abs,
    y_abs,
    inverse=True
)

events["longitude"] = lon
events["latitude"] = lat

events["depth_km"] = events["z(km)"]

events.to_csv("/home/ovsicori/Taller_Sismologia_Computacional/gamma_events.csv", index=False)
assignments.to_csv("/home/ovsicori/Taller_Sismologia_Computacional/gamma_assignments.csv", index=False)

## Plots de eventos detectados

In [ ]:
# Plot Mapa

fig = plt.figure(figsize=(10, 10))

ax = plt.axes(projection=ccrs.PlateCarree())

# Map extent
ax.set_extent(
    [-84.85, -84.35, 9.40, 9.80],
    crs=ccrs.PlateCarree()
)
"""
ax.set_extent(
    [
        events["longitude"].min() - 0.2,
        events["longitude"].max() + 0.2,
        events["latitude"].min() - 0.2,
        events["latitude"].max() + 0.2,
    ],
    crs=ccrs.PlateCarree()
)

"""

# Base map
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE, linewidth=1)
ax.add_feature(cfeature.BORDERS, linestyle=":")

# Gridlines
gl = ax.gridlines(
    draw_labels=True,
    linewidth=0.5,
    alpha=0.5
)

gl.top_labels = False
gl.right_labels = False

# EVENTS
sc = ax.scatter(
    events["longitude"],
    events["latitude"],
    c=events["depth_km"],
    s=events["num_picks"] * 12,
    cmap="viridis_r",
    edgecolors="black",
    linewidths=0.5,
    alpha=0.9,
    transform=ccrs.PlateCarree(),
    zorder=5
)

# STATIONS
ax.scatter(
    stations["lon"],
    stations["lat"],
    marker="^",
    s=90,
    color="red",
    edgecolors="black",
    linewidths=0.7,
    transform=ccrs.PlateCarree(),
    zorder=10,
    label="Stations"
)

# COLORBAR
plt.figure(figsize=(8,4))
plt.hist(events["time"], bins=20)
plt.xticks(rotation=45)
plt.title("Detected Events")
plt.show()


plt.title(
    "GaMMA Events",
    fontsize=16,
    pad=15
)

plt.legend()


plt.show()